# Tuần 5 — Trích xuất URL, Logistic Regression và XGBoost

**Nguồn đã đổi sang PhiUSIIL**, không gọi các cột của PhiUSIIL là PhishVN. Import tập trung trong cell đầu. Gói đã có mô hình huấn luyện thật và kết quả Validation; đổi RUN_TRAINING=True để chạy lại pipeline từ 2 ZIP gốc.

PhiUSIIL gốc: **0=Phishing, 1=Legitimate**. Nội bộ: **1=Phishing**. Chia theo registered domain bằng PSL offline, gồm private suffix. URL thuộc domain trong SMS được loại khỏi toàn bộ corpus URL để giảm chồng lấn giữa hai nhánh. Dữ liệu Test chỉ lưu manifest, chưa tính chỉ số.

Nguồn quy ước nhãn: https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset

In [1]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "phishguard").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "phishguard").is_dir():
    raise FileNotFoundError("Mở notebook trong thư mục notebooks của gói dự án.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from phishguard.features import extract_urls, feature_frame, infrastructure_features, LEXICAL
from phishguard.training import train_all


In [2]:
DATA_DIR = Path(r"C:\Users\ASUS\OneDrive - sad\Documents\Han\DATN")
RUN_TRAINING = False
if RUN_TRAINING:
    train_all(DATA_DIR, PROJECT_DIR)
results = json.loads((PROJECT_DIR / "reports/validation_results.json").read_text(encoding="utf-8"))
manifest = json.loads((PROJECT_DIR / "models/manifest.json").read_text(encoding="utf-8"))
print("Mô hình URL được chọn:", results["selected_url"])
print("Chia tập URL:", manifest["url_splits"])
print("Đã đánh giá Test:", results["test_evaluated"])


Mô hình URL được chọn: XGBoost_depth6
Chia tập URL: {'train': 161438, 'validation': 37975, 'test': 34017}
Đã đánh giá Test: False


## Module trích xuất dùng chung với FastAPI
Nhận http/https, www, URL trần và hxxp/[.]. Chuẩn hóa hostname, giữ hoa/thường của path/query; không tải trang web. Loại FILENAME, Title, URLSimilarityIndex, TLDLegitimateProb và các cột cần HTML/nguồn ngoài khỏi mô hình lexical. 31 đặc trưng được tính lại từ URL để bảo đảm training/serving dùng cùng định nghĩa.

In [3]:
urls = extract_urls("Vui lòng kiểm tra hxxps://example[.]com/login?ref=1 và www.example.org.")
print(urls)
display(feature_frame(urls))
print("Hạ tầng chưa có quan sát:", infrastructure_features(urls[0]))

['https://example.com/login?ref=1', 'https://www.example.org/']
Hạ tầng chưa có quan sát: {'dns_resolves': nan, 'dns_answer_count': nan, 'domain_age_days': nan, 'registration_remaining_days': nan, 'tls_valid': nan, 'dns_resolves_missing': 1, 'dns_answer_count_missing': 1, 'domain_age_days_missing': 1, 'registration_remaining_days_missing': 1, 'tls_valid_missing': 1}


,URLLength,DomainLength,IsDomainIP,TLDLength,NoOfSubDomain,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,NoOfObfuscatedChar,ObfuscationRatio,HasObfuscation,path_length,query_length,host_entropy,url_entropy,dots,hyphens,at_signs,slashes,has_punycode,has_non_ascii,has_explicit_port,credential_words,financial_words
0,31.0,11.0,0.0,3.0,0.0,23.0,0.741935,1.0,0.032258,1.0,1.0,0.0,7.0,0.225806,1.0,0.0,0.0,0.0,6.0,5.0,3.095795,4.324849,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0
1,24.0,15.0,0.0,3.0,1.0,18.0,0.750000,0.0,0.000000,0.0,0.0,0.0,6.0,0.250000,1.0,0.0,0.0,0.0,1.0,0.0,3.323231,3.855389,2.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0


## So sánh trên cùng Validation
Chuẩn hóa StandardScaler chỉ fit Train cho Logistic Regression. XGBoost dùng histogram. Lưới cố định: LR C∈{0.1,1}; XGB max_depth∈{4,6}, 240 trees. Chọn threshold theo F1; hòa thì Recall cao hơn và FPR thấp hơn. Không dùng Test chọn mô hình/ngưỡng. PR-AUC trong báo cáo được tính bằng average precision.

In [4]:
comparison = pd.DataFrame(results["url_validation"])
display(comparison[["model","precision","recall","f1","fpr","pr_auc","brier","threshold"]])
print("Kiểm soát dữ liệu:")
print(json.dumps({k: manifest[k] for k in ["source_label_mapping","sms_domain_rows_excluded_from_URL_corpus","url_conflicting_keys_removed","sms_audit"]}, ensure_ascii=False, indent=2))

Kiểm soát dữ liệu:
{
  "source_label_mapping": {
    "PhiUSIIL_0": "Phishing",
    "PhiUSIIL_1": "Legitimate"
  },
  "sms_domain_rows_excluded_from_URL_corpus": 1253,
  "url_conflicting_keys_removed": 1,
  "sms_audit": {
    "conflicting_keys_removed": 1,
    "max_cross_similarity": {
      "train_validation": 0.8796923160552979,
      "train_test": 0.8789176344871521,
      "validation_test": 0.8757433891296387
    },
    "rows": 2436,
    "splits": {
      "train": 1716,
      "validation": 376,
      "test": 344
    }
  }
}


,model,precision,recall,f1,fpr,pr_auc,brier,threshold
0,LogisticRegression_C0.1,0.977288,0.928311,0.952170,0.019033,0.978074,0.039392,0.22
1,LogisticRegression_C1.0,0.985862,0.924603,0.954250,0.011697,0.978040,0.038055,0.28
2,XGBoost_depth4,0.995219,0.947300,0.970669,0.004015,0.991737,0.024823,0.34
3,XGBoost_depth6,0.996336,0.947244,0.971170,0.003073,0.990444,0.025893,0.39


## Đặc trưng hạ tầng
`phishguard/infrastructure.py` chuyển snapshot RDAP/DNS/TLS đã đo thành domain_age_days, registration_remaining_days, dns_resolves, dns_answer_count, tls_valid. Thời điểm đo sau as_of bị loại; dữ liệu thiếu là NaN và có missing indicator. PhiUSIIL đính kèm không có snapshot tương ứng, nên mô hình đóng gói **chỉ dùng lexical**. Cần thu thập snapshot và huấn luyện lại cùng schema trước khi tuyên bố đã thử nghiệm lexical + infrastructure.

Toàn bộ mã fit nằm trong `phishguard/training.py`, mã đặc trưng trong `phishguard/features.py`. Không tự tải hoặc truy cập URL người dùng gửi.